In [1]:
import pandas as pd
import numpy as np

In [2]:
# Data Initialization, Handling Duplicates & Spaces
df = pd.read_csv('data_raw/data_capstone_RetailSales.csv', sep='|')
df.columns = df.columns.str.strip()

df.drop_duplicates(subset=['Invoice_ID', 'Store_Location', 'Product_Name', 'Transaction_Date'], inplace=True)

# Menghapus keseluruhan whitespace/spasi yang ada pada setiap kolom yang tercantum
df['Invoice_ID'] = df['Invoice_ID'].str.strip()
df['Salesperson'] = df['Salesperson'].str.strip()
df['Store_Location'] = df['Store_Location'].str.strip()
df['Product_Name'] = df['Product_Name'].str.strip()
df['Units_Sold'] = df['Units_Sold'].str.strip()
df['Revenue'] = df['Revenue'].str.strip()
df['Cost_per_Unit'] = df['Cost_per_Unit'].str.strip()
df['Transaction_Date'] = df['Transaction_Date'].str.strip()

df.head()

,Invoice_ID,Salesperson,Store_Location,Product_Name,Units_Sold,Revenue,Cost_per_Unit,Transaction_Date
0,INV-1001,Bima,Jakarta,MU Home Jersey 24/25,5 pcs,Rp 7.500.000,1 Juta,01-Mar-2024
1,inv-1002,Siti A.,BANDUNG,Arsenal Away Jersey,3 unit,"4,5 Juta",1000K,2024/03/02
2,INV-1003,NaN,surabaya,Paris SG Home 24/25,-2,Rp 3.000.000,IDR 1.000.000,03/03/2024
4,INV-1004,Andi T.,jakarta,Bayern Munich Third,10,15 Juta,1 Juta,March 04 2024
5,inv-1005,dewi p.,Bandung,Ajax Amsterdam Away,2 pcs,Rp 3.000.000,1000k,2024-03-05


In [3]:
# Standardization
df['Invoice_ID'] = df['Invoice_ID'].str.upper()
df['Salesperson'] = df['Salesperson'].str.title().replace(['NaN', 'Nan'], 'Unknown')
df['Store_Location'] = df['Store_Location'].str.title()

def standardization_product(name) :
    if isinstance(name, str) :
        name = name.replace('Home', '').replace('Away', '').replace('Third', '').replace('Jersey', '').replace('24/25', '').strip()

        if 'MU' in name or 'Man Utd' in name or 'Manchester' in name :
            return 'Manchester United'
        elif 'RM' in name or 'Real Madrid' in name :
            return 'Real Madrid'
        elif 'Paris' in name or 'PSG' in name :
            return 'Paris Saint-Germain'
        elif 'Barca' in name or 'Barcelona' in name :
            return 'Barcelona'
        elif 'Bayern' in name :
            return 'Bayern Munich'

        return ' '.join(name.split()) 


    return name

df['Product_Name'] = df['Product_Name'].apply(standardization_product)
df.head()

,Invoice_ID,Salesperson,Store_Location,Product_Name,Units_Sold,Revenue,Cost_per_Unit,Transaction_Date
0,INV-1001,Bima,Jakarta,Manchester United,5 pcs,Rp 7.500.000,1 Juta,01-Mar-2024
1,INV-1002,Siti A.,Bandung,Arsenal,3 unit,"4,5 Juta",1000K,2024/03/02
2,INV-1003,Unknown,Surabaya,Paris Saint-Germain,-2,Rp 3.000.000,IDR 1.000.000,03/03/2024
4,INV-1004,Andi T.,Jakarta,Bayern Munich,10,15 Juta,1 Juta,March 04 2024
5,INV-1005,Dewi P.,Bandung,Ajax Amsterdam,2 pcs,Rp 3.000.000,1000k,2024-03-05


In [4]:
# Numeric Extraction & Imputation:
df['Units_Sold'] = df['Units_Sold'].astype(str).str.replace('pcs|unit', '', regex=True).str.strip()
df['Units_Sold'] = df['Units_Sold'].astype(float).abs()
units_sold_median = df['Units_Sold'].median() # Variabel untuk mencari nilai median pada kolom 'Units_Sold'
df.fillna({'Units_Sold': units_sold_median}, inplace=True) # Mengisi nilai yang kosong (NaN) dengan nilai median

def removeCurrency(nominal) :
    if isinstance(nominal, str) :
        nominal = nominal.upper().strip() # Mengubah menjadi huruf kapital semua agar sama, lalu hapus spasi

        if 'JUTA' in nominal :
            # Urutannya; Depan -> Rp atau IDR, Belakang -> Juta, Tengah -> , (penanda desimal wajib menggunakan titik(.))
            nominal = nominal.replace('RP', '').replace('IDR', '').replace('JUTA', '').replace(',', '.').strip()
            return float(nominal) * 1000000
            
        elif 'K' in nominal :
            # Urutannya; Depan -> Rp atau IDR, Belakang -> Juta atau K,
            nominal = nominal.replace('RP', '').replace('IDR', '').replace('K', '').strip()
            return float(nominal) * 1000
            
        else :
            # Menambahkan penghapusan 'IDR'
            nominal = nominal.replace('RP', '').replace('IDR', '').replace('.', '').replace(',', '').strip()
            return float(nominal)

    # Jika data sudah berupa angka (float/int) dari awal, kembalikan apa adanya
    return float(nominal) if pd.notnull(nominal) else np.nan

df['Revenue'] = df['Revenue'].apply(removeCurrency)
df['Cost_per_Unit'] = df['Cost_per_Unit'].apply(removeCurrency)

df.head(10)

,Invoice_ID,Salesperson,Store_Location,Product_Name,Units_Sold,Revenue,Cost_per_Unit,Transaction_Date
0,INV-1001,Bima,Jakarta,Manchester United,5.0,7500000.0,1000000.0,01-Mar-2024
1,INV-1002,Siti A.,Bandung,Arsenal,3.0,4500000.0,1000000.0,2024/03/02
2,INV-1003,Unknown,Surabaya,Paris Saint-Germain,2.0,3000000.0,1000000.0,03/03/2024
4,INV-1004,Andi T.,Jakarta,Bayern Munich,10.0,15000000.0,1000000.0,March 04 2024
5,INV-1005,Dewi P.,Bandung,Ajax Amsterdam,2.0,3000000.0,1000000.0,2024-03-05
6,INV-1006,Bima,Surabaya,Real Madrid,4.0,6000000.0,1000000.0,06-Mar-2024
7,INV-1007,Siti A.,Jakarta,Manchester United,4.0,6000000.0,1000000.0,2024/03/07
8,INV-1008,Andi T.,Bandung,Barcelona,1.0,1500000.0,1000000.0,08/03/2024
9,INV-1009,Bima,Jakarta,Paris Saint-Germain,7.0,10500000.0,1000000.0,09-Mar-2024
10,INV-1010,Dewi P.,Surabaya,AC Milan,1.0,1500000.0,1000000.0,March 10 2024
